# 02a — Training a flood segmentation model with ml4floods
**"I want my own model."** The whole ml4floods training pipeline in a few cells, on a small regional dataset
around our common thread (the 2018 Thessaly flood):

1. From a flood map made by someone else (Copernicus EMS) to **ML-ready** data.
2. The **split by flood event** (no data leakage).
3. Large scenes → **tiles** of 256 × 256 px (precomputed for today).
4. **Train** a U-Net for a few epochs and follow its **learning curves**.
5. **Evaluate** it: choose the threshold on the validation events, then FARKADONA vs the classical methods of
   notebooks 01a/01b.

⏱ Training takes a few minutes on a GPU (Colab: *Runtime → Change runtime type → T4 GPU*) but ~20 min on a
laptop CPU. Without a GPU set `TRAIN = False` below: the notebook loads a model we trained before the
workshop with the same code and settings.

**Google Colab:** run the next cell first (~1 min). It downloads the course code, data and models and installs
ml4floods; if Colab disconnects, just run it again. On your own computer it does nothing.

In [ ]:
import sys
if "google.colab" in sys.modules:
    import subprocess
    from pathlib import Path
    COURSE = Path("/content/geomundus-ml4floods")
    if not COURSE.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/kipoju/geomundus-ml4floods", str(COURSE)], check=True)
    sys.path.insert(0, str(COURSE / "notebooks"))
    import colab_setup
    colab_setup.setup()

In [ ]:
import sys
sys.path.append(".")

import json
import os
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ml4floods
from georeader import plot, read
from georeader.geotensor import GeoTensor
from georeader.rasterio_reader import RasterioReader
from matplotlib.patches import Patch, Rectangle
from ml4floods.models.config_setup import get_default_config
from ml4floods.models.dataset_setup import get_dataset
from ml4floods.models.model_setup import get_channel_configuration_bands, get_model
from ml4floods.preprocess.tiling import WindowSize
from ml4floods.preprocess.utils import get_list_of_window_slices
from ml4floods.scripts.inference import load_inference_function
from pytorch_lightning import Trainer
from skimage.measure import label, regionprops
from course_utils import (EVENT, IN_COLAB, ROOT, binary_metrics, composite, confusion_legend, confusion_overlay,
                          edge_otsu, find_event, learning_curve_callback, ml4floods_split, mndwi,
                          plot_learning_curve, read_event, save_fig)

# Workaround for ml4floods 1.1.1 with recent PyTorch: the `verbose` argument of ReduceLROnPlateau was removed
class _ReduceLROnPlateau(torch.optim.lr_scheduler.ReduceLROnPlateau):
    def __init__(self, *args, verbose=None, **kwargs):
        super().__init__(*args, **kwargs)


torch.optim.lr_scheduler.ReduceLROnPlateau = _ReduceLROnPlateau

DATA_DIR = ROOT / "data" / "worldfloods_subset"
TRAIN = True   # False: skip training and load the model trained before the workshop

## 1. From a flood map to ML-ready data
Copernicus EMS delivers flood maps as **vector layers** (polygons of flooded areas, rivers, lakes…) mapped
by analysts on a satellite image (often radar or very-high-resolution optical). ml4floods downloads them,
finds a Sentinel-2 image of the same day and **rasterises** the polygons on its grid → a label for every
pixel. The WorldFloods ground truth has two layers:
- **clear / cloud** (cloud mask of the Sentinel-2 image),
- **land / water** (flood + permanent water).

Example: EMSR324 (Coursan, France, October 2018).

In [ ]:
EX = "EMSR324_08COURSAN_GRA_v1"
ex_dir, ex_split = find_event(EX)
ex = read_event(ex_dir, ex_split, EX)
floodmap = gpd.read_file(ex_dir / ex_split / "floodmaps" / f"{EX}.geojson")
print(floodmap.groupby(["source", "w_class"]).size())

fig, axs = plt.subplots(1, 3, figsize=(20, 6.5))
rgb = GeoTensor(composite(ex["s2"].values).transpose(2, 0, 1), transform=ex["s2"].transform, crs=ex["s2"].crs)
plot.show(rgb, ax=axs[0], add_scalebar=True)
axs[0].set_title("Sentinel-2 (true colour)")
plot.show(rgb, ax=axs[1])
colors = {"flood": "tab:red", "hydro": "tab:blue", "hydro_l": "tab:blue"}
for source, col in colors.items():
    sel = floodmap[floodmap["source"] == source]
    if len(sel):
        sel.to_crs(ex["s2"].crs).plot(ax=axs[1], color=col, linewidth=1, alpha=0.7)
axs[1].legend(handles=[Patch(color="tab:red", label="flooded area (EMS)"),
                       Patch(color="tab:blue", label="rivers / lakes (EMS)")], loc="upper right")
axs[1].set_title("Copernicus EMS flood map (vector)")
gt = ex["gt"]
gt_classes = gt.array_as_geotensor(np.where(gt[0] == 2, 3, gt[1]))   # 0 invalid, 1 land, 2 water, 3 cloud
plot.plot_segmentation_mask(gt_classes,
                            np.array([[0, 0, 0], [139, 64, 0], [0, 0, 240], [220, 220, 220]]) / 255, ax=axs[2],
                            interpretation_array=["invalid", "land", "water", "cloud"])
axs[2].set_title("ML-ready ground truth (raster)")
plt.tight_layout()
save_fig(fig, "dl_cems_to_gt")

## 2. Split by flood event
Neighbouring pixels of the same flood look alike. If tiles of the same event end up in train and test,
the test score is too optimistic (**data leakage**). So we split by **event**:
- **train**: two areas of the Thessaly flood + six small floods from around the world (WorldFloods v2 train),
- **validation**: two floods with clean reference maps, Ylitornio (Finland) and Campogalliano (Italy), to
  monitor training and choose the decision threshold,
- **test**: FARKADONA (our common thread).

⚠ Why not another area of the Thessaly flood for validation? We tried SOFADES, next to FARKADONA, but its
reference map is raw Copernicus EMS: river lines are rasterised as water and the flooded fields visible in
Sentinel-2 are missing (even MNDWI > 0 gets IoU 0.21 there). **A reference map is not the truth**: check it
before you trust a metric computed on it.

The split is a JSON file (the ml4floods format) with the Sentinel-2 image and label of every event:

In [ ]:
split = json.loads((DATA_DIR / "traintestval.json").read_text())
events = pd.DataFrame([(s, os.path.basename(f)[:-4]) for s in split for f in split[s]["S2"]],
                      columns=["split", "event"])
events

In [ ]:
# Where are the Thessaly areas? Each Sentinel-2 image at its real position, framed with the colour of its split.
# Train and test are neighbours (same flood, same day) but do not overlap; the validation floods are elsewhere.
SPLIT_COLORS = {"train": "tab:blue", "val": "tab:orange", "test": "tab:red"}
fig, ax = plt.subplots(figsize=(13, 7))
for i, (_, row) in enumerate(events[events.event.str.startswith("EMSR271")].iterrows()):
    reader = RasterioReader(str(DATA_DIR / row.split / "S2" / f"{row.event}.tif")).isel({"band": [3, 2, 1]})
    rgb = read.resize(reader.load(), resolution_dst=60)          # 10 m -> 60 m: enough for an overview map
    plot.show((rgb / 3000).clip(0, 1), ax=ax, add_scalebar=i == 0)
    xmin, ymin, xmax, ymax = rgb.bounds
    ax.add_patch(Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, edgecolor=SPLIT_COLORS[row.split], lw=3))
    ax.text(xmin + 800, ymax - 800, row.event.split("_")[1][2:], va="top", fontsize=11, fontweight="bold",
            color="white", bbox=dict(facecolor=SPLIT_COLORS[row.split], edgecolor="none", alpha=0.85, pad=2))
ax.autoscale_view()
ax.legend(handles=[Patch(facecolor="none", edgecolor=SPLIT_COLORS[s], lw=3, label=s)
                   for s in SPLIT_COLORS if s in set(events[events.event.str.startswith("EMSR271")].split)],
          loc="lower right")
ax.set_title("EMSR271 Thessaly flood (Sentinel-2, 28 Feb 2018): areas and split")
save_fig(fig, "dl_split_map")

## 3. Configuration and tiles
Everything ml4floods needs is in one configuration: data (folders, split, bands, tile size, batch size) and
model (architecture, loss weights, learning rate…). We start from the WorldFloods v2 template.

The scenes are thousands of pixels wide: we train on **tiles of 256 × 256 px**. Tiles that are mostly
cloud or no-data are dropped. That list of tiles was computed before the workshop and is loaded from
`windows_traintestval.json`.

In [ ]:
split_file = ml4floods_split(DATA_DIR)   # paths of the split and of the tiles -> absolute paths on this machine

config = get_default_config(os.path.join(os.path.dirname(ml4floods.__file__),
                                         "models/configurations/worldfloods_template_v2.json"))
config.data_params.path_to_splits = str(split_file.parent)
config.data_params.train_test_split_file = str(split_file)
config.data_params.channel_configuration = "bgriswirs"     # 6 bands: B2, B3, B4, B8, B11, B12 (as the global model of notebook 02b)
config.data_params.batch_size = 16
config.data_params.num_workers = 2 if IN_COLAB else 0
config.model_params.hyperparameters.channel_configuration = "bgriswirs"
config.model_params.hyperparameters.num_channels = 6
config.model_params.hyperparameters.model_type = "unet"   # try "simplecnn" (much smaller and faster)
config.model_params.hyperparameters.max_epochs = 5
# The template is tuned for long trainings on the whole WorldFloods (lr 1e-4, water errors weighted x20).
# With only 5 epochs that makes the model call everything water → faster learning rate, milder weight.
config.model_params.hyperparameters.lr = 1e-3
config.model_params.hyperparameters.pos_weight = [1.0, 3.0]   # [cloud, water]
config.model_params.train, config.model_params.test = True, False   # new model (test=True loads saved weights)

dataset = get_dataset(config.data_params)

In [ ]:
# Tiles of one training scene: kept (green) and dropped because of clouds / no data (red)
TILED = "EMSR271_04STEFANOVIKIO_DEL_v1"
s2_file = next(f for f in split["train"]["S2"] if TILED in f)
s2_file = str(DATA_DIR / s2_file)
kept = {(w.window.col_off, w.window.row_off) for w in dataset.train_dataset.list_of_windows if TILED in w.file_name}
all_windows = get_list_of_window_slices([s2_file], window_size=WindowSize(256, 256))

tiled = read_event(*find_event(TILED), TILED)
fig, ax = plt.subplots(figsize=(10, 8.5))
ax.imshow(composite(tiled["s2"].values))
for w in all_windows:
    ok = (w.window.col_off, w.window.row_off) in kept
    ax.add_patch(Rectangle((w.window.col_off, w.window.row_off), w.window.width, w.window.height, fill=False,
                           edgecolor="lime" if ok else "red", lw=1.2))
ax.set_title(f"{TILED}: {len(kept)} of {len(all_windows)} tiles of 256×256 px used for training")
ax.set_axis_off()
save_fig(fig, "dl_tiling")

## 4. Train
One **epoch** = the model sees every training tile once, in **batches** of 16 tiles. After each batch the
error (loss) is back-propagated and the weights are updated (stochastic gradient descent). After each
epoch the model is checked on the **validation** events, which it never trains on.

In [ ]:
EXPERIMENT = "my_unet"
model_dir = ROOT / "models" / EXPERIMENT

if TRAIN:
    torch.manual_seed(config.seed)
    model = get_model(config.model_params)
    trainer = Trainer(max_epochs=config.model_params.hyperparameters.max_epochs, accelerator="auto", logger=False,
                      enable_checkpointing=False, callbacks=[learning_curve_callback(model_dir / "learning_curve.csv")])
    model_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    trainer.fit(model, dataset)
    print(f"training time: {(time.time() - t0) / 60:.1f} min")

    # Save as the models on HuggingFace: <folder>/config.json + <folder>/model.pt
    torch.save(model.state_dict(), model_dir / "model.pt")
    (model_dir / "config.json").write_text(json.dumps(config, indent=2))
else:
    EXPERIMENT = "course_unet"   # trained before the workshop with this notebook (same settings)
    model_dir = ROOT / "models" / EXPERIMENT

### Learning curves
**Training loss**: error on the tiles the model learns from (one point per batch). **Validation loss**: error
on floods it never sees (one point per epoch). The first one always goes down; the second one tells us if
the model **generalises**.

In [ ]:
plt.figure(figsize=(9, 5))
plot_learning_curve(model_dir / "learning_curve.csv", title=f"{EXPERIMENT}: learning curves",
                    val_label="validation loss (Ylitornio + Campogalliano)")
plt.show()

❓ What happens if we keep training for longer? Before the workshop we trained the same model for 15 epochs
(its epoch 5 is the model loaded with `TRAIN = False`).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
plot_learning_curve(ROOT / "models" / "course_unet_curve" / "learning_curve.csv", ax=ax,
                    title="Same model, 15 epochs: training vs validation loss",
                    val_label="validation loss (Ylitornio + Campogalliano)")
ax.axvline(5, color="gray", ls="--"); ax.text(5.2, ax.get_ylim()[1] * 0.95, "today: 5 epochs", va="top", color="gray")
save_fig(fig, "dl_learning_curve")

## 5. Evaluate
The model gives a probability of water for every pixel; the map depends on the **threshold** we choose.
We choose it on the **validation** events and only then look at the test event (choosing it on the test
event would be leakage too).

As with `otsu_numpy` in notebook 01b, we wrap the model in a function **numpy array in → numpy array out**.
The network runs in PyTorch, which does not know about georeferencing, so the last line puts the georeference
of the input image back on the output (`array_as_geotensor`): give it a GeoTensor and you get GeoTensors back.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # the GPU if there is one
fn, cfg = load_inference_function(str(model_dir), device_name=DEVICE, max_tile_size=1024)
channels = get_channel_configuration_bands(cfg.data_params.channel_configuration)   # input bands


def unet_numpy(s2: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """P(water) and cloud mask of our U-Net for a 13-band Sentinel-2 image (band, y, x)."""
    pred_class, prob = fn(torch.tensor(np.nan_to_num(np.asarray(s2)[channels])))   # numpy -> torch -> model
    prob_water, cloud = prob[1].numpy(), pred_class.numpy() == 3
    if isinstance(s2, GeoTensor):   # torch dropped the georeference: put it back
        prob_water, cloud = s2.array_as_geotensor(prob_water), s2.array_as_geotensor(cloud)
    return prob_water, cloud


def reference(gt: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Water and valid pixels (labelled and not cloud) of a WorldFloods v2 ground truth."""
    return gt[1] == 2, (gt[1] != 0) & (gt[0] != 2)


# Validation: pool the pixels of the validation floods and pick the threshold with the best IoU
VAL_EVENTS = [os.path.basename(f)[:-4] for f in split["val"]["S2"]]
prob_val, cloud_val, ref_val, valid_val = [], [], [], []
for ev in VAL_EVENTS:
    dv = read_event(*find_event(ev), ev)
    p, c = unet_numpy(dv["s2"])
    r, v = reference(dv["gt"])
    for lst, x in zip((prob_val, cloud_val, ref_val, valid_val), (p, c, r, v)):
        lst.append(np.asarray(x).ravel())
prob_val, cloud_val, ref_val, valid_val = map(np.concatenate, (prob_val, cloud_val, ref_val, valid_val))

thresholds = np.round(np.arange(0.1, 0.95, 0.05), 2)
iou_val = [binary_metrics((prob_val > th) & ~cloud_val, ref_val, valid_val)["IoU"] for th in thresholds]
TH_WATER = float(thresholds[int(np.argmax(iou_val))])
print(f"best threshold on the validation floods: {TH_WATER} (IoU {max(iou_val):.2f})")

In [ ]:
d = read_event(*find_event(EVENT), EVENT)
prob, cloud = unet_numpy(d["s2"])   # GeoTensors on the grid of the image
ref, valid = reference(d["gt"])
print(type(prob).__name__, "| same grid as the image:", prob.same_extent(d["s2"]))
iou_test = [binary_metrics((prob > th) & ~cloud, ref, valid)["IoU"] for th in thresholds]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, iou_val, "o-", color="tab:orange", label="validation (Ylitornio + Campogalliano)")
ax.plot(thresholds, iou_test, "o-", color="tab:red", label=f"test ({EVENT.split('_')[1]})")
ax.axvline(TH_WATER, color="k", ls="--", label=f"chosen on validation: {TH_WATER}")
ax.set(xlabel="threshold on P(water)", ylabel="IoU", ylim=(0, 1), title="Choosing the threshold")
ax.grid(alpha=0.3); ax.legend()
save_fig(fig, "dl_threshold")

### Deep learning vs the classical methods (FARKADONA)
❓ **Before running:** which method will have the best IoU? Where will our U-Net still make mistakes?

Same pixels for all methods: valid and outside the clouds of the reference map.

In [ ]:
vv = d["s1"][0]
t_edge, _ = edge_otsu(vv, t0=-16.0)
methods = {   # every flood map is a GeoTensor on the same grid
    "MNDWI > 0 (optical)": mndwi(d["s2"]) > 0,
    "Edge Otsu VV + HAND < 15 m (SAR)": (vv < t_edge) & (d["hand"] < 15),
    f"our U-Net, P(water) > {TH_WATER} (deep learning)": (prob > TH_WATER) & ~cloud,
}
table = pd.DataFrame({name: binary_metrics(p, ref, valid) for name, p in methods.items()}).T.round(2)
table

In [ ]:
def confusion_panels(axs, items):
    for ax, (name, p) in zip(axs, items):
        m = table.loc[name]
        ax.imshow(confusion_overlay(p, ref, valid))
        ax.set_title(f"{name}\nP {m['precision']:.2f} · R {m['recall']:.2f} · IoU {m['IoU']:.2f}", fontsize=12)
        ax.set_axis_off()
    confusion_legend(axs[0])


fig, axs = plt.subplots(1, 2, figsize=(14, 6.5))     # classical only (slide "Where rules break")
confusion_panels(axs, list(methods.items())[:2])
plt.tight_layout()
save_fig(fig, "dl_rules_break")
plt.close(fig)

fig, axs = plt.subplots(1, 3, figsize=(20, 6.5))
confusion_panels(axs, methods.items())
plt.tight_layout()
save_fig(fig, "dl_vs_classical")

### Where does our model fail?
Zoom on its largest false alarm.

In [ ]:
ours = methods[f"our U-Net, P(water) > {TH_WATER} (deep learning)"]
fp = label((ours & ~ref & valid).values)   # skimage: plain numpy
r0, c0, r1, c1 = max(regionprops(fp), key=lambda r: r.area).bbox
pad = 150
win = (slice(max(r0 - pad, 0), r1 + pad), slice(max(c0 - pad, 0), c1 + pad))
fig, axs = plt.subplots(1, 3, figsize=(18, 6.5))
axs[0].imshow(composite(d["s2"].values, ("B4", "B3", "B2"))[win]); axs[0].set_title("Sentinel-2 true colour")
axs[1].imshow(composite(d["s2"].values, ("B11", "B8", "B4"), 4500)[win]); axs[1].set_title("SWIR / NIR / Red")
axs[2].imshow(confusion_overlay(ours, ref, valid)[win]); axs[2].set_title("our U-Net: errors")
for ax in axs:
    ax.set_axis_off()
confusion_legend(axs[2])
plt.tight_layout()
save_fig(fig, "dl_ours_failure")

Dark in every band, next to a cloud: a **cloud shadow**. Our 8 training floods barely contain cloud shadows,
so the model never learned that they are not water → **a model fails on what it has not seen**. The global
model of notebook 02b was trained on ~500 flood maps from all over the world.

## 6. Your turn: change one thing
Go back to section 3, change **one** setting, re-run sections 3–5 and compare with your neighbour:
- `max_epochs`: 2 vs 10,
- `model_type`: `"unet"` vs `"simplecnn"`,
- `channel_configuration`: `"bgriswirs"` vs `"rgbi"` (only visible + NIR, e.g. for a cheap satellite)
  (change it in both `data_params` and `model_params.hyperparameters`) → also set `num_channels = 4`.

**Take-aways**
- ml4floods turns flood maps made by others into ML-ready data, and a model into a flood map, in a few lines.
- Split by event; tile large scenes; evaluate on floods the model has never seen.
- A small model trained in minutes on a few floods already beats the rules near its training region, but it
  fails on what it has not seen (cloud shadows). What would you need to use it on the next flood in another
  continent? → notebook 02b.